# पाठ 18: क्रिप्टोग्राफिक रसीदों के साथ AI एजेंट्स को सुरक्षित करना

## हाथों-हाथ नोटबुक

यह नोटबुक चार अभ्यासों के माध्यम से चलता है:

1. एक एजेंट टूल कॉल के लिए **अपनी पहली रसीद पर हस्ताक्षर करें** और उसकी पुष्टि करें।
2. रसीद के साथ **छेड़छाड़ करें** और सत्यापन विफल होने को देखें।
3. **तीन-रसीद श्रृंखला बनाएं** और श्रृंखला की अखंडता की पुष्टि करें।
4. Microsoft एजेंट फ्रेमवर्क टूल कॉल को **रैप करें** ताकि हर क्रिया एक रसीद जारी करे।

सभी क्रिप्टोग्राफिक प्रिमिटिव्स अच्छी तरह से मेंटेन की गई लाइब्रेरीज़ से आयात किए गए हैं (`pynacl` Ed25519 के लिए, `jcs` RFC 8785 कैनोनिकल JSON के लिए, `hashlib` SHA-256 के लिए Python मानक लाइब्रेरी से)। रसीद लॉजिक स्वयं एक सिंपल Python है जिसे आप पढ़ और संशोधित कर सकते हैं।

सेल्स को क्रम में चलाएं। प्रत्येक अनुभाग छोटा और आत्मनिर्भर है।


## सेटअप

दो निर्भरताएँ इंस्टॉल करें। दोनों के पास उदार लाइसेंस हैं (Apache-2.0 / MIT)।


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## सहायक उपयोगिताएँ

ये दो सहायक बेस64url एन्कोडिंग (पैडिंग के बिना) और arbitrary ऑब्जेक्ट्स के SHA-256 हैशिंग को संभालते हैं। वे बाकी नोटबुक को रिसीट लॉजिक पर केंद्रित रखते हैं।


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## अनुभाग 1: अपनी पहली रसीद पर हस्ताक्षर करें

कल्पना करें कि हमारे एजेंट के लिए **Contoso Travel** ने अभी एक ग्राहक के लिए सिडनी से लॉस एंजेल्स की उड़ानें देखीं। हम इस टूल कॉल के बारे में गेटवे के बयान को रिकॉर्ड करना चाहते हैं ताकि भविष्य में कोई ऑडिटर इसे अलग से पिन किए गए जारीकर्ता कुंजी के तहत सत्यापित कर सके।

### चरण 1.1: एक हस्ताक्षर कुंजी जनरेट करें

उत्पादन में, एजेंट की हस्ताक्षर कुंजी हार्डवेयर सुरक्षा मॉड्यूल (HSM), Azure Key Vault, या इसी तरह के संरक्षित भंडारण में रहती है। इस पाठ के लिए हम मेमोरी में एक नई कुंजी बनाते हैं।


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### चरण 1.2: रसीद पेलोड बनाएं

पेलोड में वह सब कुछ शामिल होता है जिसे हम रसीद में प्रमाणित करना चाहते हैं: किसने कार्रवाई की, कौन सा टूल, किन तर्कों के साथ, क्या वापस आया, किस नीति के तहत, और कब। हम तर्कों और परिणाम को इनलाइन शामिल करने के बजाय उनका हैश बनाते हैं ताकि रसीद संवेदनशील सामग्री को लीक न करे।


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### चरण 1.3: रसीद पर हस्ताक्षर करें और उसे एकत्रित करें

तीन चरण:

1. JCS का उपयोग करके पेलोड को रेगुलराइज़ करें ताकि दो इम्प्लीमेंटेशन जो एक ही तार्किक रसीद बनाते हैं, वे बाइट्स में बिल्कुल समान हों।
2. Ed25519 प्राइवेट की के साथ सीधे रेगुलराइज़ किए गए बाइट्स पर हस्ताक्षर करें। PureEdDSA संदेश को आंतरिक रूप से हैश करता है, इसलिए अतिरिक्त प्री-हैश प्रोटोकॉल को बदल देगा।

फिर हस्ताक्षर को अंतिम रसीद बनाने के लिए मूल पेलोड से जोड़ा जाता है।


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### चरण 1.4: रसीद सत्यापित करें  

सत्यापन प्रक्रिया को उलट देता है। हम हस्ताक्षर को हटाते हैं, प्रामाणिक बाइट्स को पुनः गणना करते हैं, और जारीकर्ता की सार्वजनिक कुंजी के खिलाफ हस्ताक्षर की जांच करते हैं।  

सत्यापन ऑफ़लाइन होता है: कॉल करने के लिए कोई सेवा नहीं और कोई नेटवर्क एक्सेस नहीं। इसके लिए रसीद के अलावा एक चीज़ की आवश्यकता होती है: जारीकर्ता की सार्वजनिक कुंजी, जिसे ऑडिटर पहले से भरोसेमंद चैनल के माध्यम से प्राप्त करता है, जैसे जारीकर्ता द्वारा प्रकाशित एक JWK सेट या पहले से पिन किया गया मान। रसीद के अंदर `public_key` फ़ील्ड यह भूमिका नहीं निभा सकती, क्योंकि जो भी रसीद लिखता है वही वह फ़ील्ड भी लिखता है। अनुभाग 2 यह दिखाता है कि यह धोखाधड़ी को कैसे रोकता है।  


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


आपको `Receipt is valid: True`, `Pre-hashed receipt valid: False` और `Malformed receipts rejected: True` दिखाई देने चाहिए। सकारात्मक मामला यह साबित करता है कि डायरेक्ट-JCS साइनिंग पथ काम करता है; नकारात्मक नियंत्रण हस्ताक्षर-स्कोप नियम को प्रोस के बजाय निष्पादन योग्य बनाता है; अंतिम पंक्ति गलत इनपुट को अपवाद उठाने के बजाय `False` लौटाते हुए दिखाती है।


## अनुभाग 2: रसीद में छेड़छाड़ करें

रसीदों का पूरा मकसद यह है कि वे छेड़छाड़-स्पष्ट हों। चलिए इसे साबित करते हैं।

हम रसीद के ठीक एक अक्षर को बदलेंगे और देखेंगे कि सत्यापन फेल हो जाता है।


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### अभी क्या हुआ?

जब हमने `policy_id` को बदला, तो कैनोनिकल बाइट्स बदल गए। सिग्नेचर (जो मूल कैनोनिकल बाइट्स पर था) अब मेल नहीं खाता। सत्यापन सही ढंग से `False` लौटाता है।

कोई तरीका नहीं है कि साइन किए गए फ़ील्ड को बदला जाए, यानी `signature` ऑब्जेक्ट के बाहर कुछ भी, और फिर भी रसीद जारीकर्ता की सार्वजनिक कुंजी के खिलाफ सत्यापित हो, जब तक कि हमलावर के पास जारीकर्ता की निजी कुंजी न हो। `signature` ऑब्जेक्ट खुद हस्ताक्षरित नहीं है: `sig` सिग्नेचर है, और `verify_receipt` `public_key` की उपेक्षा करता है, इसलिए केवल उस फ़ील्ड को संपादित करने से कुछ नहीं बदलता। जब तक निजी कुंजी कीी वॉल्ट में होती है और सत्यापक जारीकर्ता की स्वतंत्र रूप से प्रकाशित कुंजी का उपयोग करते हैं, तब तक साइन किए गए फ़ील्ड में कोई भी बदलाव छिपाना असंभव है। यह स्थिति महत्वपूर्ण है: अगली सेल में एक संशोधित रसीद है जो अलग कुंजी के खिलाफ पूरी तरह से सत्यापित होती है।

खुद कोशिश करें: ऊपर दी गई सेल में `tool_name` या `agent_id` या `timestamp` को संशोधित करें और फिर से चलाएं। साइन किए गए किसी भी फ़ील्ड में बदलाव एक अमान्य रसीद उत्पन्न करता है।


### हस्ताक्षर उस छेड़छाड़ को नहीं पकड़ता

ऊपर वाला हमलावर फिर से हस्ताक्षर नहीं करता था। एक सावधान व्यक्ति ऐसा करता: रसीद को बदलता,  
स्वयं बनाए गए कुंजी से उसे हस्ताक्षरित करता, और उस कुंजी को  
`public_key` फ़ील्ड में डालता। परिणामस्वरूप रसीद की हस्ताक्षर पूरी तरह उसके  
सामग्री के लिए वास्तविक होती है।  

इसे चलाएँ और देखें कि दो जांचों में से कौन इसे पकड़ता है।  


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


आपको `True` फिर `False` दिखाई देना चाहिए। तीनों `assert` पंक्तियाँ इसे एक
जांच बनाती हैं: अगर फर्जी कोई भी समय जारीकर्ता की कुंजी के खिलाफ सत्यापित होती है तो सेल एक त्रुटि उठाता है।


फर्जी कार्य आंतरिक रूप से संगत है। इसका हस्ताक्षर वास्तव में इसके
सामग्री और इसके साथ जोड़ी गई कुंजी से मेल खाता है, इसलिए एक सत्यापनकर्ता जो रसीद से कुंजी पढ़ता है
इसे मान्य के रूप में रिपोर्ट करता है। रसीद में कुछ भी गलत नहीं दिखता, क्योंकि
इसमें कुछ भी असंगत नहीं है।

इसे केवल तब ही पकड़ पाता है जब इसे ऐसी कुंजी के खिलाफ जांचा जाता है जिस पर हमलावर का कोई
प्रभाव नहीं हो सकता। इसलिए `verify_receipt` कुंजी को तर्क के रूप में लेता है: 
एक ऐसी फ़ंक्शन जिसका हस्ताक्षर `verify_receipt(receipt)` है सुरक्षित रूप से नहीं लिखा जा सकता,
क्योंकि वह केवल उस कुंजी तक पहुँच सकता है जो हमलावर द्वारा प्रदान की गई हो।

वही रसीद `sample_receipts/04_resigned_receipt.json` के रूप में सहेजी जाती है यदि आप
इसे नोटबुक के बाहर जांचना चाहते हैं।


## अनुभाग 3: रसीदों को एक साथ जोड़ना

एक एकल रसीद एक क्रिया की सुरक्षा करता है। अधिकांश एजेंट कई क्रियाएं करते हैं। पूरे अनुक्रम को छेड़छाड़-सूचक बनाने के लिए, हम प्रत्येक रसीद को पिछले रसीद के हैश को नए रसीद के पेलोड में शामिल करके पिछले रसीद से जोड़ते हैं।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

यदि कोई प्रस्तुत किए गए अनुक्रम के अंदर कोई रसीद हटा देता है या पुनःक्रमित करता है, तो अगली कड़ी टूट जाती है। ट्रंकशन का पता लगाना या हस्ताक्षरकर्ता नियंत्रित प्रत्यय पुनर्लेखन के लिए भी एक पिन की गई वर्तमान हेड, बाहरी चेकपॉइंट, या प्रमाणित पॉपुलेशन प्रतिबद्धता की आवश्यकता होती है।


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


अब मध्य रसीद के साथ छेड़छाड़ करके श्रृंखला को तोड़ें और पुनः सत्यापित करें। छेड़ी हुई रसीद अपनी हस्ताक्षर जांच में विफल हो जाती है, और अगली रसीद अपनी श्रृंखला-लिंक जांच में विफल हो जाती है (क्योंकि इसका `previous_receipt_hash` अब संशोधित मध्य रसीद के हैश से मेल नहीं खाता)।


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


रसीद 0 अभी भी सत्यापित होती है (इसमें कोई संशोधन नहीं किया गया है और यह किसी पूर्ववर्ती पर निर्भर नहीं है)। रसीद 1 अपनी हस्ताक्षर जांच में विफल होती है क्योंकि हमने `tool_args_hash` को बदल दिया। रसीद 2 अपनी श्रृंखला-लिंक जांच में विफल होती है क्योंकि इसका `previous_receipt_hash` मूल (अब-संशोधित) रसीद 1 के खिलाफ गणना किया गया था।

भले ही एक हमलावर संशोधित रसीद 1 को फिर से हस्ताक्षरित करे, नया हस्ताक्षर जारीकर्ता की कुंजी के खिलाफ विफल हो जाता है जब तक कि उनके पास जारीकर्ता की निजी कुंजी न हो (उनकी अपनी कुंजी के साथ पुनः हस्ताक्षर विफल होता है, जैसा कि अनुभाग 2 ने दिखाया), और रसीद 2 में श्रृंखला-लिंक असंगति फिर भी छेड़छाड़ को उजागर करेगी। परिवर्तन को छिपाने के लिए, हमलावर को संशोधन बिंदु से आगे की हर रसीद को फिर से हस्ताक्षरित करना होगा, जिसके लिए जारीकर्ता की निजी कुंजी होना आवश्यक है।


## अनुभाग 4: रसीद पर हस्ताक्षर के साथ एजेंट टूल कॉल को रैप करें

एक वास्तविक तैनाती में, आप नहीं चाहते कि हर एजेंट लेखक को `make_receipt` कॉल करना याद रहे। आप चाहते हैं कि प्रत्येक टूल इनवोकेशन के लिए रसीद पर हस्ताक्षर स्वचालित हो।

यहां सबसे सरल पैटर्न है: एक रैपर क्लास जो किसी भी कॉल करने योग्य टूल फ़ंक्शन को लेती है और उसकी एक रसीद-जारी करने वाली संस्करण लौटाती है। यह किसी भी एजेंट फ्रेमवर्क के लिए अनुकूल है, जिसमें Microsoft एजेंट फ्रेमवर्क (`agent_framework.foundry`) भी शामिल है।

यदि आपके पास Microsoft Foundry परियोजना सेटअप नहीं है, तो नीचे का स्थानीय मॉक पैटर्न को फिर भी प्रदर्शित करता है।


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft एजेंट फ्रेमवर्क के साथ एकीकरण

ऊपर दिया गया `ReceiptedTool` रैपर फ्रेमवर्क-एग्नोस्टिक है। इसे Microsoft एजेंट फ्रेमवर्क के साथ बनाए गए एजेंट में उपयोग करने के लिए, रैप्ड फ़ंक्शन को एक टूल के रूप में पंजीकृत करें। एक स्केच (आप मॉक को एक वास्तविक Microsoft Foundry टूल पंजीकरण से बदलेंगे):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

एजेंट फ्रेमवर्क को रसीदों के बारे में कुछ भी जानने की जरूरत नहीं है। रसीद साइनिंग टूल के चारों ओर लिपटी हुई है, फ्रेमवर्क में जोड़ी नहीं गई है। यही तरीका है कि आप मौजूदा एजेंट कोड में पुनःलेखन किए बिना प्रामाणिकता जोड़ते हैं।


## पुनर्कथन और स्ट्रेच चैलेंज

आपके पास है:

- एक Ed25519 की जोड़ी जनरेट की।
- एक एजेंट टूल कॉल के लिए रसीद बनाई और साइन की।
- केवल जारीकर्ता की सार्वजनिक कुंजी का उपयोग करके ऑफ़लाइन रसीद सत्यापित की, जो रसीद से स्वतंत्र रूप से प्राप्त की गई थी।
- रसीद के साथ छेड़छाड़ की और सत्यापन विफलता देखी।
- तीन रसीदों की एक हैश-चेन वाली क्रमिक श्रृंखला बनाई।
- श्रृंखला के बीच में छेड़छाड़ की और दोनों सिग्नेचर विफलता और चेन-लिंक विफलता देखी।
- एक टूल फ़ंक्शन को स्वचालित रसीद हस्ताक्षर के साथ लपेटा।

**स्ट्रेच चैलेंज।** रसीद स्कीमा को एक `request_id` फ़ील्ड (वितरित ट्रेसिंग के लिए UUID) के साथ बढ़ाएं। इसे शामिल करने के लिए `make_receipt` को अपडेट करें, और पुष्टि करें कि रसीदें अभी भी अंत से अंत तक सत्यापित होती हैं। फिर साइनिंग के बाद फ़ील्ड को संशोधित करें और सत्यापन विफल होने की पुष्टि करें। यह आपको यह आंतरिक रूप से समझने के लिए बाध्य करता है कि कैनोनिकल एन्कोडिंग का हर बाइट सिग्नेचर में कैसे योगदान करता है।

**महत्वपूर्ण सीमा।** पिन की और चेन संदर्भ के तहत, रसीदें सिग्नेचर अट्रिब्यूशन, सामग्री अखंडता, और प्रस्तुत पूर्ववर्ती से लिंक साबित करती हैं। वे निष्पादन, स्रोत सत्य, भरोसेमंद समय, पूर्णता, नीति मूल्यांकन, या सही होने का प्रमाण नहीं देतीं। रसीदें एक आधार हैं। शासन वह प्रणाली है जो आप इसके ऊपर बनाते हैं।

उस सीमा को ध्यान में रखते हुए पाठ की README को फिर से पढ़ें। रसीदों के साथ टीमों की सबसे सामान्य गलती यह मानना है कि "हमारे पास रसीदें हैं" मतलब "हम शासित हैं।" ऐसा नहीं है। रसीदें एजेंट के व्यवहार को ऑडिट योग्य बनाती हैं। वे इसे सही नहीं बनातीं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
